In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")
import pickle

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
res = pd.read_csv("sample_submission.csv")

In [ ]:
train.drop("id", axis=1, inplace=True)

In [ ]:
data_info = {}

data_info["Driver"] = train.Driver.value_counts(normalize=True)
data_info["Compound"] = train.Compound.value_counts(normalize=True)
data_info["Race"] = train.Race.value_counts(normalize=True)

for i in data_info["Driver"].keys():
    train[f"Driver_{i}"] = (train.Driver == i).astype("int8")
for i in data_info["Compound"].keys():
    train[f"Compound_{i}"] = (train.Compound == i).astype("int8")
for i in data_info["Race"].keys():
    train[f"Race_{i}"] = (train.Race == i).astype("int8")

train.drop(["Driver", "Compound", "Race"], axis=1, inplace=True)

for i in data_info["Driver"].keys():
    test[f"Driver_{i}"] = (test.Driver == i).astype("int8")
for i in data_info["Compound"].keys():
    test[f"Compound_{i}"] = (test.Compound == i).astype("int8")
for i in data_info["Race"].keys():
    test[f"Race_{i}"] = (test.Race == i).astype("int8")

test.drop(["Driver", "Compound", "Race"], axis=1, inplace=True)

In [ ]:
train["Year"] = (2026 - train["Year"]).astype("int8")
test["Year"] = (2026 - test["Year"]).astype("int8")

In [ ]:
train[["LapNumber", "Stint", "TyreLife", "LapTime (s)", "RaceProgress"]] = np.log1p(train[["LapNumber", "Stint", "TyreLife", "LapTime (s)", "RaceProgress"]])
test[["LapNumber", "Stint", "TyreLife", "LapTime (s)", "RaceProgress"]] = np.log1p(test[["LapNumber", "Stint", "TyreLife", "LapTime (s)", "RaceProgress"]])


In [ ]:
train[['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change']].hist(bins=50, figsize=(20, 20));

In [ ]:
train = train[(train["LapTime_Delta"] > -70) & (train["LapTime_Delta"] < 70)]
test = test[(test["LapTime_Delta"] > -70) & (test["LapTime_Delta"] < 70)]


In [ ]:


train = train[train["LapTime (s)"] < 5.5]
test = test[test["LapTime (s)"] < 5.5]

In [ ]:
train = train[train["Cumulative_Degradation"] < 500]
test = test[test["Cumulative_Degradation"] < 500]


In [ ]:
x, y = train.drop("PitNextLap", axis=1), train["PitNextLap"].values
# x_test, y_test = test.drop("PitNextLap", axis=1), test["PitNextLap"].values
test.drop("id", axis=1, inplace=True)


In [ ]:
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x

In [ ]:
train.columns[:15]

In [ ]:
ssc = StandardScaler()
msc = MinMaxScaler()

x[['LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'Position_Change']] = ssc.fit_transform(x[['LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'Position_Change']])
x[['Year', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'RaceProgress']] = msc.fit_transform(x[['Year', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'RaceProgress']])

test[['LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'Position_Change']] = ssc.transform(test[['LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'Position_Change']])
test[['Year', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'RaceProgress']] = msc.transform(test[['Year', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'RaceProgress']])

In [ ]:
train

In [ ]:
test

In [ ]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=100,
    task_type='CPU'
)

# Train
model.fit(x, y)

# Predict probabilities
y_pred_proba = model.predict_proba(test)[:, 1]
y_train_pred_proba = model.predict_proba(x)[:, 1]

# Predict classes
y_pred = model.predict(test)
y_train_pred = model.predict(x)

# Metrics
print("Accuracy:", accuracy_score(y, y_train_pred))
print("ROC AUC:", roc_auc_score(y, y_train_pred_proba))

In [ ]:
data_info["model"] = model

In [ ]:
with open("f1_pitstop_info.pck", "wb") as f:
    pickle.dump(data_info, f)

In [ ]:
res["class"] = y_pred

In [ ]:
res